<a href="https://colab.research.google.com/github/cristianzucconi2-web/iatoarts/blob/main/iatoarts2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installiamo mido per gestire i file MIDI
!pip install mido
import random
import mido
from mido import Message, MidiFile, MidiTrack

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.7 MB/s eta 0:00:00


2. La Fitness (Le Regole del Giudice)
Oltre alla distanza (fondamentale per la fluidità), ti propongo di aggiungere questi criteri per dare "carattere" alla musica:

Regola della Tonalità (Opzionale ma consigliata): Anche se vogliamo "tutte le note", la musica di solito suona meglio se segue una scala. Possiamo decidere che se la nota appartiene a una scala (es. Minore Melodica per un suono più cupo o Maggiore per uno solare) riceve un bonus.

Regola della Densità (Varietà): Se il computer usa solo 2 note per tutta la canzone, gli togliamo punti. Vogliamo che esplori le note che gli abbiamo dato.

Regola del "Ritmo" (Silenzio): In musica le pause sono importanti. Possiamo inserire un "gene" speciale che rappresenta il silenzio. Una melodia senza pause è asfissiante.

Direzionalità: Possiamo premiare le melodie che non restano sempre piatte, ma che "salgono" o "scendono" con un senso logico.

3. Evitare la Ripetizione (Diversità)
Per avere una melodia sempre diversa ogni volta che schiacci "Play":

Useremo una Mutazione più alta: così il computer non si accontenta della prima cosa carina che trova, ma continua a sperimentare.

Fitness Dinamica: Possiamo dire al programma: "Se assomigli troppo alla melodia che hai generato l'ultima volta, ti tolgo punti".

----------------------------------------


-------------------------------
1. La Struttura Orizzontale (Tempo)
Un paio di minuti sono tanti per un algoritmo genetico se cerchiamo di ottimizzare tutto in un colpo solo (il computer farebbe fatica a gestire una lista di 500 note).

La Soluzione: Dividiamo la canzone in blocchi (es. 8 blocchi da 15 secondi l'uno).

L'algoritmo genera il "Blocco A", poi il "Blocco B" e così via, incollandoli alla fine. Questo rende la melodia coerente ma varia.

2. La Struttura Verticale (Strumenti)
Per avere più strumenti sopra l'altro, il nostro Cromosoma non sarà più una lista singola, ma una matrice (una tabella).

Traccia 1 (Piano): Melodia principale.

Traccia 2 (Basso): Note lunghe e profonde.

Traccia 3 (Archi/Pad): Accordi di sottofondo.

Come funziona la Fitness: Dovrà controllare non solo che le note di uno strumento siano belle, ma che la nota del Piano "stia bene" con quella del Basso nello stesso momento.

3. Il "Dataset" delle Note (Range 36-96)
Useremo tutto il range, ma con una distinzione:

Il Basso pescherà note tra 36 e 48.

Il Piano pescherà note tra 60 e 84.

Gli Archi pescheranno note tra 48 e 72.

In [3]:
# --- PARAMETRI DEL PROGETTO ---
NOTE_BASSO = list(range(36, 49))    # Note profonde
NOTE_PIANO = list(range(60, 85))    # Note medie/alte
DURATA_BLOCCO = 32                 # 32 note per ogni strumento per blocco
NUMERO_STRUMENTI = 2               # Iniziamo con Piano + Basso per non complicare troppo
POPOLAZIONE_SIZE = 50              # Numero di tentativi per ogni blocco
GENERAZIONI = 100                  # Quante volte facciamo evolvere ogni blocco

print(f"Setup pronto: Genereremo {NUMERO_STRUMENTI} tracce contemporanee.")

Setup pronto: Genereremo 2 tracce contemporanee.


In [4]:
def crea_individuo_multitraccia():
    # Creiamo la traccia del Piano (note medie/alte)
    traccia_piano = [random.choice(NOTE_PIANO) for _ in range(DURATA_BLOCCO)]

    # Creiamo la traccia del Basso (note profonde)
    traccia_basso = [random.choice(NOTE_BASSO) for _ in range(DURATA_BLOCCO)]

    # L'individuo è l'unione delle due: [ [piano...], [basso...] ]
    return [traccia_piano, traccia_basso]

# Testiamo la creazione
test_individuo = crea_individuo_multitraccia()
print("Esempio di un individuo (le prime 5 note di ogni traccia):")
print(f"Piano: {test_individuo[0][:5]}")
print(f"Basso: {test_individuo[1][:5]}")

Esempio di un individuo (le prime 5 note di ogni traccia):
Piano: [77, 71, 78, 78, 64]
Basso: [36, 42, 48, 43, 48]


In [5]:
def calcola_fitness_multitraccia(individuo):
    piano = individuo[0]
    basso = individuo[1]
    voto = 0

    for i in range(DURATA_BLOCCO):
        # --- 1. Regole del Piano (Orizzontale) ---
        if i < DURATA_BLOCCO - 1:
            dist_p = abs(piano[i] - piano[i+1])
            if dist_p <= 4: voto += 10  # Premio fluidità
            if dist_p > 12: voto -= 20  # Penalità salto enorme

        # --- 2. Regole del Basso (Orizzontale) ---
        if i < DURATA_BLOCCO - 1:
            dist_b = abs(basso[i] - basso[i+1])
            if dist_b == 0: voto += 15  # Il basso che tiene la stessa nota è buono
            if dist_b > 7: voto -= 30   # Il basso non deve saltare quasi mai

        # --- 3. Regola Verticale (Piano + Basso insieme) ---
        # Calcoliamo l'intervallo tra la nota del piano e quella del basso in quel momento
        intervallo = abs(piano[i] - basso[i])
        if intervallo % 12 == 0:
            voto += 20 # Ottava perfetta: suona da Dio!
        elif intervallo % 12 in [7, 5, 4]:
            voto += 10 # Quinta, quarta o terza: suonano bene.

    return voto

print(f"Voto del test: {calcola_fitness_multitraccia(test_individuo)}")

Voto del test: 135


In [6]:
def evoluzione_multitraccia(popolazione_attuale):
    # 1. Classifica: chi è il miglior compositore?
    popolazione_ordinata = sorted(popolazione_attuale, key=calcola_fitness_multitraccia, reverse=True)

    # 2. Selezione: teniamo i migliori 10
    migliori = popolazione_ordinata[:10]
    nuova_generazione = migliori.copy()

    while len(nuova_generazione) < POPOLAZIONE_SIZE:
        padre = random.choice(migliori)
        madre = random.choice(migliori)

        # CROSSOVER: Tagliamo entrambe le tracce nello stesso punto
        punto = random.randint(1, DURATA_BLOCCO - 1)

        figlio_piano = padre[0][:punto] + madre[0][punto:]
        figlio_basso = padre[1][:punto] + madre[1][punto:]

        figlio = [figlio_piano, figlio_basso]

        # MUTAZIONE: 15% di probabilità di cambiare una nota (o al piano o al basso)
        if random.random() < 0.15:
            traccia_da_mutare = random.randint(0, 1)
            indice = random.randint(0, DURATA_BLOCCO - 1)
            if traccia_da_mutare == 0:
                figlio[0][indice] = random.choice(NOTE_PIANO)
            else:
                figlio[1][indice] = random.choice(NOTE_BASSO)

        nuova_generazione.append(figlio)

    return nuova_generazione

In [7]:
canzone_completa_piano = []
canzone_completa_basso = []
NUMERO_BLOCCHI = 4

print("Inizio composizione dell'opera...")

for b in range(NUMERO_BLOCCHI):
    # Creiamo una nuova popolazione per ogni blocco
    popolazione = [crea_individuo_multitraccia() for _ in range(POPOLAZIONE_SIZE)]

    for g in range(GENERAZIONI):
        popolazione = evoluzione_multitraccia(popolazione)

    # Prendiamo il migliore di questo blocco
    vincitore = popolazione[0]
    canzone_completa_piano.extend(vincitore[0])
    canzone_completa_basso.extend(vincitore[1])

    print(f"Blocco {b+1}/{NUMERO_BLOCCHI} completato. Fitness: {calcola_fitness_multitraccia(vincitore)}")

print("\nComposizione terminata!")

Inizio composizione dell'opera...
Blocco 1/4 completato. Fitness: 880
Blocco 2/4 completato. Fitness: 790
Blocco 3/4 completato. Fitness: 810
Blocco 4/4 completato. Fitness: 875

Composizione terminata!


In [8]:
def salva_opera_completa(piano, basso, nome_file="opera_magna.mid"):
    mid = MidiFile()

    # --- TRACCIA 1: PIANO ---
    track1 = MidiTrack()
    mid.tracks.append(track1)
    track1.append(mido.MetaMessage('track_name', name='Piano Melodia', time=0))
    # Impostiamo lo strumento (Program Change 0 = Acoustic Grand Piano)
    track1.append(mido.Message('program_change', program=0, time=0))

    for nota in piano:
        track1.append(mido.Message('note_on', note=nota, velocity=90, time=0))
        track1.append(mido.Message('note_off', note=nota, velocity=90, time=480))

    # --- TRACCIA 2: BASSO ---
    track2 = MidiTrack()
    mid.tracks.append(track2)
    track2.append(mido.MetaMessage('track_name', name='Basso Accompagnamento', time=0))
    # Impostiamo lo strumento (Program Change 32 = Acoustic Bass)
    track2.append(mido.Message('program_change', program=32, time=0))

    # Le note del basso suonano insieme a quelle del piano
    for nota in basso:
        track2.append(mido.Message('note_on', note=nota, velocity=70, time=0))
        track2.append(mido.Message('note_off', note=nota, velocity=70, time=480))

    mid.save(nome_file)
    print(f"L'opera è pronta! Scarica '{nome_file}' dalla cartella a sinistra.")

# Salviamo la nostra composizione
salva_opera_completa(canzone_completa_piano, canzone_completa_basso)

L'opera è pronta! Scarica 'opera_magna.mid' dalla cartella a sinistra.


ISTRUZIONI PER CAPIRE SE FUNZIONA






Cosa fare adesso in GarageBand:
Scarica opera_magna.mid da Colab.

Trascinalo in GarageBand.

Importante: GarageBand ti chiederà se vuoi "Importare il tempo" o "Separare le tracce". Di' di Sì.

Ti appariranno due righe:

Sulla prima (Piano) metti uno strumento come un Grand Piano o un Synth Lead.

Sulla seconda (Basso) metti un Acoustic Bass o un Deep Synth Bass.

Schiaccia Play e goditi il risultato della tua IA!
